In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

import joblib

print("Libraries imported successfully! ✅")

Libraries imported successfully! ✅


In [2]:
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("Random seed set successfully! ✅")

Random seed set successfully! ✅


In [3]:
cities = {
    "Mumbai": (19.0760, 72.8777),
    "Delhi": (28.6139, 77.2090),
    "Bengaluru": (12.9716, 77.5946),
    "Kolkata": (22.5726, 88.3639),
    "Chennai": (13.0827, 80.2707)
}

START_DATE = "2021-01-01"
END_DATE = "2023-12-31"

print("Cities selected:")
for city in cities:
    print("-", city)

print("\nDate range:")
print(START_DATE, "to", END_DATE)

Cities selected:
- Mumbai
- Delhi
- Bengaluru
- Kolkata
- Chennai

Date range:
2021-01-01 to 2023-12-31


In [4]:
import requests

all_weather_data = []

for city, coordinates in cities.items():

    latitude, longitude = coordinates

    print(f"Downloading data for {city}...")

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "precipitation,"
            "surface_pressure,"
            "wind_speed_10m,"
            "wind_direction_10m"
        ),
        "timezone": "Asia/Kolkata"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:

        data = response.json()["hourly"]

        city_df = pd.DataFrame(data)

        city_df["city"] = city

        all_weather_data.append(city_df)

        print(f"✅ {city} downloaded successfully!")

    else:
        print(f"❌ Failed to download {city}")
        print(response.status_code)

print("\nDownload process complete!")

✅ Mumbai downloaded successfully!
✅ Delhi downloaded successfully!
✅ Bengaluru downloaded successfully!
✅ Kolkata downloaded successfully!
✅ Chennai downloaded successfully!

Download process complete!


In [5]:
weather_df = pd.concat(
    all_weather_data,
    ignore_index=True
)

print("All city datasets combined! ✅")

print("\nDataset shape:")
print(weather_df.shape)

print("\nColumns:")
print(weather_df.columns.tolist())

display(weather_df.head())

All city datasets combined! ✅

Dataset shape:
(131400, 8)

Columns:
['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'city']


,time,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,city
0,2021-01-01T00:00,23.1,75,0.0,1010.8,10.2,352,Mumbai
1,2021-01-01T01:00,22.8,74,0.0,1010.5,7.2,360,Mumbai
2,2021-01-01T02:00,22.3,77,0.0,1009.8,6.2,350,Mumbai
3,2021-01-01T03:00,21.8,82,0.0,1009.6,4.7,351,Mumbai
4,2021-01-01T04:00,21.4,84,0.0,1009.5,4.7,9,Mumbai


In [12]:
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Sort properly by city and time
weather_df = weather_df.sort_values(
    by=["city", "time"]
).reset_index(drop=True)

print("DATASET INFORMATION")
print("=" * 50)

print("\nShape:")
print(weather_df.shape)

print("\nMissing values:")
print(weather_df.isnull().sum())

print("\nData types:")
print(weather_df.dtypes)

print("\nFirst 5 rows:")
display(weather_df.head())

DATASET INFORMATION

Shape:
(131400, 8)

Missing values:
time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
surface_pressure        0
wind_speed_10m          0
wind_direction_10m      0
city                    0
dtype: int64

Data types:
time                    datetime64[ns]
temperature_2m                 float64
relative_humidity_2m             int64
precipitation                  float64
surface_pressure               float64
wind_speed_10m                 float64
wind_direction_10m               int64
city                            object
dtype: object

First 5 rows:


,time,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,city
0,2021-01-01 00:00:00,16.9,96,0.0,913.3,14.1,87,Bengaluru
1,2021-01-01 01:00:00,16.7,98,0.0,912.8,14.2,83,Bengaluru
2,2021-01-01 02:00:00,17.3,94,0.0,912.5,16.3,82,Bengaluru
3,2021-01-01 03:00:00,17.5,92,0.0,912.1,16.5,80,Bengaluru
4,2021-01-01 04:00:00,17.3,92,0.0,911.8,14.4,77,Bengaluru


In [11]:
# Create time-based features

forecast_df = combined_df.copy()

forecast_df["hour"] = forecast_df["time"].dt.hour
forecast_df["day"] = forecast_df["time"].dt.day
forecast_df["month"] = forecast_df["time"].dt.month
forecast_df["day_of_year"] = forecast_df["time"].dt.dayofyear

print("Time features created successfully! ✅")

print("\nNew columns:")
print(forecast_df.columns.tolist())

forecast_df.head()

Forecast dataset created successfully! ✅

Shape:
(131400, 8)


,time,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,city
0,2021-01-01 00:00:00,16.9,96,0.0,913.3,14.1,87,Bengaluru
1,2021-01-01 01:00:00,16.7,98,0.0,912.8,14.2,83,Bengaluru
2,2021-01-01 02:00:00,17.3,94,0.0,912.5,16.3,82,Bengaluru
3,2021-01-01 03:00:00,17.5,92,0.0,912.1,16.5,80,Bengaluru
4,2021-01-01 04:00:00,17.3,92,0.0,911.8,14.4,77,Bengaluru


In [13]:
# Create time-based features

forecast_df["hour"] = forecast_df["time"].dt.hour
forecast_df["day"] = forecast_df["time"].dt.day
forecast_df["month"] = forecast_df["time"].dt.month
forecast_df["day_of_year"] = forecast_df["time"].dt.dayofyear

print("Time-based features added successfully! ✅")

print("\nCurrent columns:")
print(forecast_df.columns.tolist())

display(forecast_df.head())

Time-based features added successfully! ✅

Current columns:
['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'city', 'hour', 'day', 'month', 'day_of_year']


,time,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,city,hour,day,month,day_of_year
0,2021-01-01 00:00:00,16.9,96,0.0,913.3,14.1,87,Bengaluru,0,1,1,1
1,2021-01-01 01:00:00,16.7,98,0.0,912.8,14.2,83,Bengaluru,1,1,1,1
2,2021-01-01 02:00:00,17.3,94,0.0,912.5,16.3,82,Bengaluru,2,1,1,1
3,2021-01-01 03:00:00,17.5,92,0.0,912.1,16.5,80,Bengaluru,3,1,1,1
4,2021-01-01 04:00:00,17.3,92,0.0,911.8,14.4,77,Bengaluru,4,1,1,1


In [14]:
import numpy as np

# Set random seed so results remain reproducible
np.random.seed(42)

# Create a copy before generating forecast values
forecast_training_df = forecast_df.copy()


# --------------------------------------------------
# SIMULATED TEMPERATURE FORECAST
# --------------------------------------------------

forecast_training_df["forecast_temperature"] = (
    forecast_training_df["temperature_2m"]
    + np.random.normal(0, 2.5, len(forecast_training_df))
)


# --------------------------------------------------
# SIMULATED HUMIDITY FORECAST
# --------------------------------------------------

forecast_training_df["forecast_humidity"] = (
    forecast_training_df["relative_humidity_2m"]
    + np.random.normal(0, 8, len(forecast_training_df))
)

# Keep humidity within realistic limits
forecast_training_df["forecast_humidity"] = (
    forecast_training_df["forecast_humidity"]
    .clip(0, 100)
)


# --------------------------------------------------
# SIMULATED PRECIPITATION FORECAST
# --------------------------------------------------

forecast_training_df["forecast_precipitation"] = (
    forecast_training_df["precipitation"]
    + np.random.normal(0, 2, len(forecast_training_df))
)

# Rainfall cannot be negative
forecast_training_df["forecast_precipitation"] = (
    forecast_training_df["forecast_precipitation"]
    .clip(lower=0)
)


# --------------------------------------------------
# SIMULATED PRESSURE FORECAST
# --------------------------------------------------

forecast_training_df["forecast_pressure"] = (
    forecast_training_df["surface_pressure"]
    + np.random.normal(0, 3, len(forecast_training_df))
)


# --------------------------------------------------
# SIMULATED WIND SPEED FORECAST
# --------------------------------------------------

forecast_training_df["forecast_wind_speed"] = (
    forecast_training_df["wind_speed_10m"]
    + np.random.normal(0, 3, len(forecast_training_df))
)

# Wind speed cannot be negative
forecast_training_df["forecast_wind_speed"] = (
    forecast_training_df["forecast_wind_speed"]
    .clip(lower=0)
)


print("Simulated forecast values created successfully! ✅")

print("\nNew dataset shape:")
print(forecast_training_df.shape)

print("\nNew forecast columns:")
print([
    "forecast_temperature",
    "forecast_humidity",
    "forecast_precipitation",
    "forecast_pressure",
    "forecast_wind_speed"
])

display(forecast_training_df.head())

Simulated forecast values created successfully! ✅

New dataset shape:
(131400, 17)

New forecast columns:
['forecast_temperature', 'forecast_humidity', 'forecast_precipitation', 'forecast_pressure', 'forecast_wind_speed']


,time,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,city,hour,day,month,day_of_year,forecast_temperature,forecast_humidity,forecast_precipitation,forecast_pressure,forecast_wind_speed
0,2021-01-01 00:00:00,16.9,96,0.0,913.3,14.1,87,Bengaluru,0,1,1,1,18.141785,93.958102,0.712065,917.504592,16.843914
1,2021-01-01 01:00:00,16.7,98,0.0,912.8,14.2,83,Bengaluru,1,1,1,1,16.354339,100.000000,0.000000,910.709476,12.327111
2,2021-01-01 02:00:00,17.3,94,0.0,912.5,16.3,82,Bengaluru,2,1,1,1,18.919221,99.612803,0.000000,916.009089,13.191843
3,2021-01-01 03:00:00,17.5,92,0.0,912.1,16.5,80,Bengaluru,3,1,1,1,21.307575,100.000000,0.000000,912.119996,16.184800
4,2021-01-01 04:00:00,17.3,92,0.0,911.8,14.4,77,Bengaluru,4,1,1,1,16.714617,84.437177,0.000000,910.907686,15.488537


In [17]:
import numpy as np

# Make sure we are working with the correct dataframe
forecast_df = weather_df.copy()

# Recreate time features if needed
forecast_df["hour"] = forecast_df["time"].dt.hour
forecast_df["day"] = forecast_df["time"].dt.day
forecast_df["month"] = forecast_df["time"].dt.month
forecast_df["day_of_year"] = forecast_df["time"].dt.dayofyear

# Recreate simulated forecast values
np.random.seed(42)

forecast_df["forecast_temperature"] = (
    forecast_df["temperature_2m"]
    + np.random.normal(0, 2.5, len(forecast_df))
)

forecast_df["forecast_humidity"] = (
    forecast_df["relative_humidity_2m"]
    + np.random.normal(0, 8, len(forecast_df))
).clip(0, 100)

forecast_df["forecast_precipitation"] = (
    forecast_df["precipitation"]
    + np.random.normal(0, 1.5, len(forecast_df))
).clip(0)

forecast_df["forecast_pressure"] = (
    forecast_df["surface_pressure"]
    + np.random.normal(0, 5, len(forecast_df))
)

forecast_df["forecast_wind_speed"] = (
    forecast_df["wind_speed_10m"]
    + np.random.normal(0, 4, len(forecast_df))
).clip(0)

# Calculate forecast errors
forecast_df["temperature_error"] = (
    forecast_df["forecast_temperature"]
    - forecast_df["temperature_2m"]
)

forecast_df["humidity_error"] = (
    forecast_df["forecast_humidity"]
    - forecast_df["relative_humidity_2m"]
)

forecast_df["precipitation_error"] = (
    forecast_df["forecast_precipitation"]
    - forecast_df["precipitation"]
)

forecast_df["pressure_error"] = (
    forecast_df["forecast_pressure"]
    - forecast_df["surface_pressure"]
)

forecast_df["wind_speed_error"] = (
    forecast_df["forecast_wind_speed"]
    - forecast_df["wind_speed_10m"]
)

print("Forecast values and error features created successfully! ✅")

print("\nDataset shape:", forecast_df.shape)

print("\nNew columns:")
new_columns = [
    "forecast_temperature",
    "forecast_humidity",
    "forecast_precipitation",
    "forecast_pressure",
    "forecast_wind_speed",
    "temperature_error",
    "humidity_error",
    "precipitation_error",
    "pressure_error",
    "wind_speed_error"
]

for col in new_columns:
    print("-", col)

display(
    forecast_df[
        [
            "temperature_2m",
            "forecast_temperature",
            "temperature_error",
            "relative_humidity_2m",
            "forecast_humidity",
            "humidity_error"
        ]
    ].head()
)

Forecast values and error features created successfully! ✅

Dataset shape: (131400, 22)

New columns:
- forecast_temperature
- forecast_humidity
- forecast_precipitation
- forecast_pressure
- forecast_wind_speed
- temperature_error
- humidity_error
- precipitation_error
- pressure_error
- wind_speed_error


,temperature_2m,forecast_temperature,temperature_error,relative_humidity_2m,forecast_humidity,humidity_error
0,16.9,18.141785,1.241785,96,93.958102,-2.041898
1,16.7,16.354339,-0.345661,98,100.000000,2.000000
2,17.3,18.919221,1.619221,94,99.612803,5.612803
3,17.5,21.307575,3.807575,92,100.000000,8.000000
4,17.3,16.714617,-0.585383,92,84.437177,-7.562823


In [16]:
print("CURRENT FORECAST_DF COLUMNS:")
print(forecast_df.columns.tolist())

print("\nShape:", forecast_df.shape)

CURRENT FORECAST_DF COLUMNS:
['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'city', 'hour', 'day', 'month', 'day_of_year']

Shape: (131400, 12)


In [18]:
print("Current forecast_df shape:", forecast_df.shape)

print("\nCurrent columns:")
print(forecast_df.columns.tolist())

print("\nMissing expected columns:")

expected_columns = [
    "forecast_temperature",
    "forecast_humidity",
    "forecast_precipitation",
    "forecast_pressure",
    "forecast_wind_speed",
    "temperature_error",
    "humidity_error",
    "precipitation_error",
    "pressure_error",
    "wind_speed_error"
]

for col in expected_columns:
    if col not in forecast_df.columns:
        print("❌", col)

Current forecast_df shape: (131400, 22)

Current columns:
['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'city', 'hour', 'day', 'month', 'day_of_year', 'forecast_temperature', 'forecast_humidity', 'forecast_precipitation', 'forecast_pressure', 'forecast_wind_speed', 'temperature_error', 'humidity_error', 'precipitation_error', 'pressure_error', 'wind_speed_error']

Missing expected columns:


In [19]:
# Create absolute forecast errors

forecast_df["abs_temperature_error"] = (
    forecast_df["temperature_error"].abs()
)

forecast_df["abs_humidity_error"] = (
    forecast_df["humidity_error"].abs()
)

forecast_df["abs_precipitation_error"] = (
    forecast_df["precipitation_error"].abs()
)

forecast_df["abs_pressure_error"] = (
    forecast_df["pressure_error"].abs()
)

forecast_df["abs_wind_speed_error"] = (
    forecast_df["wind_speed_error"].abs()
)

absolute_error_columns = [
    "abs_temperature_error",
    "abs_humidity_error",
    "abs_precipitation_error",
    "abs_pressure_error",
    "abs_wind_speed_error"
]

print("Absolute forecast errors created successfully! ✅")

print("\nNew columns:")
for col in absolute_error_columns:
    print("-", col)

display(
    forecast_df[
        [
            "temperature_error",
            "abs_temperature_error",
            "humidity_error",
            "abs_humidity_error",
            "precipitation_error",
            "abs_precipitation_error"
        ]
    ].head()
)

Absolute forecast errors created successfully! ✅

New columns:
- abs_temperature_error
- abs_humidity_error
- abs_precipitation_error
- abs_pressure_error
- abs_wind_speed_error


,temperature_error,abs_temperature_error,humidity_error,abs_humidity_error,precipitation_error,abs_precipitation_error
0,1.241785,1.241785,-2.041898,2.041898,0.534049,0.534049
1,-0.345661,0.345661,2.000000,2.000000,0.000000,0.000000
2,1.619221,1.619221,5.612803,5.612803,0.000000,0.000000
3,3.807575,3.807575,8.000000,8.000000,0.000000,0.000000
4,-0.585383,0.585383,-7.562823,7.562823,0.000000,0.000000


In [20]:
# Define forecast bust thresholds

TEMP_BUST_THRESHOLD = 5
HUMIDITY_BUST_THRESHOLD = 15
PRECIP_BUST_THRESHOLD = 5
PRESSURE_BUST_THRESHOLD = 12
WIND_BUST_THRESHOLD = 8


# Create forecast bust label

forecast_df["forecast_bust"] = (
    (forecast_df["abs_temperature_error"] >= TEMP_BUST_THRESHOLD) |
    (forecast_df["abs_humidity_error"] >= HUMIDITY_BUST_THRESHOLD) |
    (forecast_df["abs_precipitation_error"] >= PRECIP_BUST_THRESHOLD) |
    (forecast_df["abs_pressure_error"] >= PRESSURE_BUST_THRESHOLD) |
    (forecast_df["abs_wind_speed_error"] >= WIND_BUST_THRESHOLD)
).astype(int)


print("Forecast Bust labels created successfully! 🚨")

print("\nForecast Bust Distribution:")
print(forecast_df["forecast_bust"].value_counts())

print("\nPercentage Distribution:")
print(
    forecast_df["forecast_bust"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Forecast Bust labels created successfully! 🚨

Forecast Bust Distribution:
forecast_bust
0    112457
1     18943
Name: count, dtype: int64

Percentage Distribution:
forecast_bust
0    85.58
1    14.42
Name: proportion, dtype: float64


In [21]:
# Define the input features for Model 2

feature_columns = [

    # Forecast values
    "forecast_temperature",
    "forecast_humidity",
    "forecast_precipitation",
    "forecast_pressure",
    "forecast_wind_speed",

    # Time information
    "hour",
    "day",
    "month",
    "day_of_year"
]

target_column = "forecast_bust"


print("MODEL 2 FEATURES")
print("=" * 50)

print("\nInput features:")
for feature in feature_columns:
    print("-", feature)

print("\nTarget:")
print("-", target_column)

print("\nTotal features:", len(feature_columns))

MODEL 2 FEATURES

Input features:
- forecast_temperature
- forecast_humidity
- forecast_precipitation
- forecast_pressure
- forecast_wind_speed
- hour
- day
- month
- day_of_year

Target:
- forecast_bust

Total features: 9


In [22]:
# Create feature matrix and target variable

X = forecast_df[feature_columns]
y = forecast_df[target_column]

print("FEATURE MATRIX AND TARGET CREATED! ✅")

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

FEATURE MATRIX AND TARGET CREATED! ✅

X shape: (131400, 9)
y shape: (131400,)

Target distribution:
forecast_bust
0    112457
1     18943
Name: count, dtype: int64


In [23]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("TRAIN / TEST SPLIT COMPLETED! ✅")

print("\nTraining data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

TRAIN / TEST SPLIT COMPLETED! ✅

Training data:
X_train: (105120, 9)
y_train: (105120,)

Testing data:
X_test: (26280, 9)
y_test: (26280,)

Training target distribution:
forecast_bust
0    85.58
1    14.42
Name: proportion, dtype: float64

Testing target distribution:
forecast_bust
0    85.58
1    14.42
Name: proportion, dtype: float64


In [24]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit on training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Only transform test data
X_test_scaled = scaler.transform(X_test)


print("FEATURE SCALING COMPLETED! ✅")

print("\nScaled training data shape:")
print(X_train_scaled.shape)

print("\nScaled testing data shape:")
print(X_test_scaled.shape)

print("\nFirst scaled training row:")
print(X_train_scaled[0])

FEATURE SCALING COMPLETED! ✅

Scaled training data shape:
(105120, 9)

Scaled testing data shape:
(26280, 9)

First scaled training row:
[-0.17609855  0.05257433 -0.61775963  0.4244139  -0.18894882  1.08266818
  0.14813046 -1.31297841 -1.2812075 ]


In [25]:
from sklearn.ensemble import RandomForestClassifier

# Create Model 2

forecast_bust_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


# Train the model

forecast_bust_model.fit(
    X_train_scaled,
    y_train
)

print("MODEL 2 TRAINING COMPLETED! 🎉")

MODEL 2 TRAINING COMPLETED! 🎉


In [26]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Make predictions

y_pred = forecast_bust_model.predict(X_test_scaled)


# Calculate performance metrics

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


print("MODEL 2 PERFORMANCE")
print("=" * 50)

print("\nPrecision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


# Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)


# Detailed Classification Report

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Reliable Forecast",
            "Forecast Bust"
        ]
    )
)

MODEL 2 PERFORMANCE

Precision: 0.41329384666008556
Recall: 0.3314858801794669
F1 Score: 0.36789689513766843

Confusion Matrix:
[[20708  1783]
 [ 2533  1256]]

Classification Report:

                   precision    recall  f1-score   support

Reliable Forecast       0.89      0.92      0.91     22491
    Forecast Bust       0.41      0.33      0.37      3789

         accuracy                           0.84     26280
        macro avg       0.65      0.63      0.64     26280
     weighted avg       0.82      0.84      0.83     26280



In [29]:
# Get probability scores for Forecast Bust

y_prob = forecast_bust_model.predict_proba(X_test_scaled)[:, 1]

print("FORECAST BUST PROBABILITY SCORES")
print("=" * 50)

print("Minimum probability:", y_prob.min())
print("Maximum probability:", y_prob.max())
print("Average probability:", y_prob.mean())

FORECAST BUST PROBABILITY SCORES
Minimum probability: 0.1033974036809704
Maximum probability: 0.9076581646716873
Average probability: 0.38230094907407863


In [28]:
# Check available model-like variables

for name, value in list(globals().items()):
    if hasattr(value, "predict") or hasattr(value, "predict_proba"):
        print(name, "→", type(value))

RandomForestClassifier → <class 'abc.ABCMeta'>
forecast_bust_model → <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [30]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import pandas as pd

# Try multiple probability thresholds

thresholds = np.linspace(0.10, 0.90, 100)

results = []

for threshold in thresholds:

    # Convert probabilities into predictions
    y_pred_threshold = (y_prob >= threshold).astype(int)

    # Calculate performance
    precision = precision_score(y_test, y_pred_threshold, zero_division=0)
    recall = recall_score(y_test, y_pred_threshold, zero_division=0)
    f1 = f1_score(y_test, y_pred_threshold, zero_division=0)

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

# Create results dataframe
threshold_results = pd.DataFrame(results)

# Sort by best F1 score
top_thresholds = threshold_results.sort_values(
    by="f1_score",
    ascending=False
)

print("TOP 10 THRESHOLDS BY F1 SCORE")
print("=" * 60)

display(top_thresholds.head(10))

# Get best threshold
best_row = top_thresholds.iloc[0]

best_threshold_model2 = best_row["threshold"]

print("\n🏆 BEST THRESHOLD:", best_threshold_model2)
print("Precision:", best_row["precision"])
print("Recall:", best_row["recall"])
print("F1 Score:", best_row["f1_score"])

TOP 10 THRESHOLDS BY F1 SCORE


,threshold,precision,recall,f1_score
45,0.463636,0.343616,0.443917,0.387379
44,0.455556,0.327018,0.473740,0.386937
46,0.471717,0.359050,0.415149,0.385067
43,0.447475,0.308130,0.500132,0.381326
47,0.479798,0.371270,0.387437,0.379181
42,0.439394,0.292023,0.532330,0.377150
48,0.487879,0.387179,0.361837,0.374079
41,0.431313,0.277684,0.565849,0.372546
49,0.495960,0.406152,0.341515,0.371039
40,0.423232,0.265461,0.600422,0.368153



🏆 BEST THRESHOLD: 0.4636363636363636
Precision: 0.3436159346271706
Recall: 0.44391660068619687
F1 Score: 0.3873790879778904


In [31]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Apply the optimized threshold

y_pred_optimized = (
    y_prob >= best_threshold_model2
).astype(int)


# Calculate final metrics

precision = precision_score(y_test, y_pred_optimized)
recall = recall_score(y_test, y_pred_optimized)
f1 = f1_score(y_test, y_pred_optimized)

cm = confusion_matrix(y_test, y_pred_optimized)


print("FINAL OPTIMIZED MODEL 2 PERFORMANCE")
print("=" * 55)

print("\nPrecision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_optimized,
        target_names=[
            "Reliable Forecast",
            "Forecast Bust"
        ]
    )
)

FINAL OPTIMIZED MODEL 2 PERFORMANCE

Precision: 0.3436159346271706
Recall: 0.44391660068619687
F1 Score: 0.3873790879778904

Confusion Matrix:
[[19278  3213]
 [ 2107  1682]]

Classification Report:

                   precision    recall  f1-score   support

Reliable Forecast       0.90      0.86      0.88     22491
    Forecast Bust       0.34      0.44      0.39      3789

         accuracy                           0.80     26280
        macro avg       0.62      0.65      0.63     26280
     weighted avg       0.82      0.80      0.81     26280



In [32]:
import joblib

# Save trained Model 2
joblib.dump(
    forecast_bust_model,
    "forecast_bust_model.pkl"
)

# Save scaler
joblib.dump(
    scaler,
    "forecast_bust_scaler.pkl"
)

# Save feature list
joblib.dump(
    feature_columns,
    "forecast_bust_features.pkl"
)

# Save optimized threshold
joblib.dump(
    best_threshold_model2,
    "forecast_bust_threshold.pkl"
)

print("All Model 2 files saved successfully! ✅")

print("\nSaved files:")
print("1. forecast_bust_model.pkl")
print("2. forecast_bust_scaler.pkl")
print("3. forecast_bust_features.pkl")
print("4. forecast_bust_threshold.pkl")

All Model 2 files saved successfully! ✅

Saved files:
1. forecast_bust_model.pkl
2. forecast_bust_scaler.pkl
3. forecast_bust_features.pkl
4. forecast_bust_threshold.pkl


In [40]:
import joblib

# Load the saved Model 2 components

forecast_bust_model = joblib.load("forecast_bust_model.pkl")

forecast_scaler = joblib.load("forecast_bust_scaler.pkl")

forecast_features = joblib.load("forecast_bust_features.pkl")

best_threshold = joblib.load("forecast_bust_threshold.pkl")


print("Model 2 components loaded successfully! ✅")

print("\nModel:", type(forecast_bust_model))
print("Scaler:", type(forecast_scaler))
print("Features:", forecast_features)
print("Threshold:", best_threshold)

Model 2 components loaded successfully! ✅

Model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Scaler: <class 'sklearn.preprocessing._data.StandardScaler'>
Features: ['forecast_temperature', 'forecast_humidity', 'forecast_precipitation', 'forecast_pressure', 'forecast_wind_speed', 'hour', 'day', 'month', 'day_of_year']
Threshold: 0.4636363636363636


In [41]:
# Define the exact features used during Model 2 training

forecast_features = [
    "forecast_temperature",
    "forecast_humidity",
    "forecast_precipitation",
    "forecast_pressure",
    "forecast_wind_speed",
    "hour",
    "day",
    "month",
    "day_of_year"
]


# Create prediction function for Forecast Bust Detection

def predict_forecast_bust(
    forecast_temperature,
    forecast_humidity,
    forecast_precipitation,
    forecast_pressure,
    forecast_wind_speed,
    hour,
    day,
    month,
    day_of_year
):

    # Create input dataframe
    input_data = pd.DataFrame([{
        "forecast_temperature": forecast_temperature,
        "forecast_humidity": forecast_humidity,
        "forecast_precipitation": forecast_precipitation,
        "forecast_pressure": forecast_pressure,
        "forecast_wind_speed": forecast_wind_speed,
        "hour": hour,
        "day": day,
        "month": month,
        "day_of_year": day_of_year
    }])

    # Ensure exact feature order
    input_data = input_data[forecast_features]

    # Scale input
    input_scaled = forecast_scaler.transform(input_data)

    # Get Forecast Bust probability
    bust_probability = forecast_bust_model.predict_proba(
        input_scaled
    )[:, 1][0]

    # Apply optimized threshold
    prediction = int(
        bust_probability >= best_threshold
    )

    # Convert to readable status
    if prediction == 1:
        status = "FORECAST BUST LIKELY ⚠️"
    else:
        status = "FORECAST APPEARS RELIABLE ✅"

    return {
        "status": status,
        "forecast_bust": prediction,
        "bust_probability": float(bust_probability),
        "threshold_used": float(best_threshold)
    }


print("Forecast Bust prediction function created successfully! ✅")

Forecast Bust prediction function created successfully! ✅


In [42]:
normal_test = predict_forecast_bust(
    forecast_temperature=28,
    forecast_humidity=70,
    forecast_precipitation=1,
    forecast_pressure=1012,
    forecast_wind_speed=12,
    hour=12,
    day=15,
    month=6,
    day_of_year=166
)

print("RELIABLE FORECAST TEST")
print("-" * 40)

for key, value in normal_test.items():
    print(f"{key}: {value}")

RELIABLE FORECAST TEST
----------------------------------------
status: FORECAST APPEARS RELIABLE ✅
forecast_bust: 0
bust_probability: 0.3788618575019715
threshold_used: 0.4636363636363636


In [43]:
bust_test = predict_forecast_bust(
    forecast_temperature=42,
    forecast_humidity=95,
    forecast_precipitation=40,
    forecast_pressure=980,
    forecast_wind_speed=80,
    hour=18,
    day=20,
    month=7,
    day_of_year=201
)

print("FORECAST BUST TEST")
print("-" * 40)

for key, value in bust_test.items():
    print(f"{key}: {value}")

FORECAST BUST TEST
----------------------------------------
status: FORECAST BUST LIKELY ⚠️
forecast_bust: 1
bust_probability: 0.7116811819416634
threshold_used: 0.4636363636363636
